# 🎬 Clip Mining with Qwen3.5-4B - Kaggle GPU

**Purpose:** Detect viral clip candidates using **Qwen3.5-4B** on **Kaggle's FREE GPU**

**Model:** `Qwen/Qwen3.5-4B` (4B parameters)

**GPU:** P100 or V100 (16GB VRAM)

---

## ⚠️ IMPORTANT: Before Running

### 1. Enable GPU
   - Click **"More"** (⋮) → **"Accelerator"**
   - Select **"GPU T4 x2"** or **"P100 GPU"**

### 2. Add Hugging Face Token
   - Click **"Add secret"** (left panel)
   - Name: `HF_TOKEN`
   - Value: `hf_xxxxxxxxxxxxxxxxxxxxx`
   - Get token from: https://huggingface.co/settings/tokens

### 3. Upload Data
   - Create **Dataset** with your `chunks.json`
   - Or upload directly in notebook

---

## Step 1: Install Dependencies

In [ ]:
# Install latest Transformers from GitHub
print("Installing latest transformers...")
!pip install -q -U git+https://github.com/huggingface/transformers.git

# Install linear attention support
print("Installing flash attention...")
!pip install -q einops

# Install core dependencies
!pip install -q torch accelerate huggingface_hub

# Verify
print("\n" + "="*60)
print("INSTALLED PACKAGES")
print("="*60)
import transformers
import torch
print(f'transformers: {transformers.__version__}')
print(f'torch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print("="*60)

## Step 2: Setup Hugging Face Authentication

In [ ]:
import os
from huggingface_hub import login, HfApi

print("="*60)
print("HUGGING FACE AUTHENTICATION")
print("="*60)

# Get HF token from Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    print("✓ Found HF_TOKEN in Kaggle Secrets")
except Exception as e:
    print(f"❌ HF_TOKEN not found: {e}")
    print("\nPlease add HF_TOKEN to Kaggle Secrets:")
    print("  1. Click 'Add secret' (left panel)")
    print("  2. Name: HF_TOKEN")
    print("  3. Value: your_hf_token")
    raise SystemExit("HF_TOKEN required")

# Login to Hugging Face
try:
    login(token=hf_token)
    api = HfApi()
    user = api.whoami(token=hf_token)
    print(f"✓ Logged in as: {user['name']}")
    
    # Check Qwen3.5-4B access
    try:
        api.model_info("Qwen/Qwen3.5-4B", token=hf_token)
        print(f"✓ Access confirmed: Qwen/Qwen3.5-4B")
    except Exception as e:
        print(f"⚠️ Model access issue: {e}")
        
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    raise

print("="*60)

## Step 3: Check GPU

In [ ]:
import torch

print("="*60)
print("GPU STATUS")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"✓ GPU: {gpu_name}")
    print(f"  Memory: {gpu_memory:.1f} GB")
    
    # Check GPU type
    if "P100" in gpu_name:
        print(f"  ✓ NVIDIA P100 - Excellent for Qwen3.5-4B!")
    elif "V100" in gpu_name:
        print(f"  ✓ NVIDIA V100 - Perfect!")
    elif "T4" in gpu_name:
        print(f"  ✓ NVIDIA T4 - Good!")
    
    print(f"  Expected VRAM usage: ~10-12 GB")
    print(f"  Available VRAM: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU detected!")
    print("Please enable GPU: More (⋮) → Accelerator → GPU")
    raise SystemExit("GPU required")

print("="*60)

## Step 4: Load Qwen3.5-4B Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-4B"

VALID_TOPICS = [
    "Charity", "Oppression", "Dua", "Mercy", "Death",
    "Tawakkul", "Sabr", "Afterlife", "Faith", "Prayer",
    "Quran", "Hadith", "Other"
]

print(f"Loading model: {MODEL_NAME}")
print("Using HF authentication: ✓ Yes")
print("Using flash attention: ✓ Enabled")
print("This may take 2-3 minutes...\n")

try:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        token=hf_token,
        trust_remote_code=True
    )
    
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        token=hf_token,
        attn_implementation="flash_attention_2"
    )
    
    print(f"✓ Model loaded successfully!")
    print(f"  Device: {model.device}")
    print(f"  VRAM used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")
    
except Exception as e:
    print(f"❌ Failed to load model: {e}")
    raise

## Step 5: Load Transcripts

In [ ]:
import json

print("Loading transcripts...")

# Try different paths (Kaggle datasets)
possible_paths = [
    "/kaggle/input/my-dataset/chunks.json",
    "/kaggle/working/chunks.json",
    "chunks.json"
]

chunks = None
used_path = None

for path in possible_paths:
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            chunks = data if isinstance(data, list) else data.get('chunks', [])
            used_path = path
            break

if not chunks:
    print("❌ No chunks.json found!")
    print("\nUpload chunks.json:")
    print("  1. Create new dataset or use existing")
    print("  2. Upload chunks.json to dataset")
    print("  3. Add dataset to this notebook")
    raise SystemExit("chunks.json required")
else:
    print(f"✓ Loaded {len(chunks)} chunks from {used_path}")

## Step 6: Process with Qwen3.5-4B

In [ ]:
# Keywords for filtering
IMPORTANT_KEYWORDS = [
    "allah", "prophet", "quran", "hadith", "islam",
    "faith", "prayer", "zakat", "sadaqah", "charity",
    "dua", "mercy", "paradise", "hellfire", "death"
]

EMOTIONAL_KEYWORDS = [
    "love", "mercy", "fear", "hope", "heart", "soul",
    "paradise", "hellfire", "forgiveness"
]

def has_important_content(text):
    text_lower = text.lower()
    return any(kw in text_lower for kw in IMPORTANT_KEYWORDS)

def calculate_clip_score(text, topic):
    score = 5
    text_lower = text.lower()
    if any(w in text_lower for w in EMOTIONAL_KEYWORDS): score += 2
    if any(w in text_lower for w in ['allah', 'prophet', 'quran']): score += 1
    if any(w in text_lower for w in ['remember', 'learn', 'lesson']): score += 1
    if topic in ['Charity', 'Dua', 'Mercy', 'Patience']: score += 1
    return min(score, 10)

def classify_topic_llm(text):
    prompt = f"""Classify topic: {', '.join(VALID_TOPICS[:8])}
Text: {text[:500]}
Return topic only."""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            temperature=0.3,
            do_sample=False
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    topic = response.strip().title()
    return topic if topic in VALID_TOPICS else "Other"

def process_chunk(chunk):
    text = chunk.get("text", "")
    if not has_important_content(text):
        return {**chunk, "topic": "Other", "emotion_score": 3, "clip_candidate": False, "skipped": True}
    
    topic = classify_topic_llm(text)
    emotion_score = calculate_clip_score(text, topic)
    clip_candidate = (topic != "Other" and emotion_score >= 7)
    
    return {**chunk, "topic": topic, "emotion_score": emotion_score, "clip_candidate": clip_candidate, "skipped": False}


# Process all chunks
print("\n" + "="*60)
print("PROCESSING WITH QWEN3.5-4B")
print("="*60)

processed_chunks = []
skipped_count = 0

for i, chunk in enumerate(chunks):
    result = process_chunk(chunk)
    processed_chunks.append(result)
    if result.get("skipped"): skipped_count += 1
    
    if (i + 1) % 50 == 0:
        candidates = sum(1 for c in processed_chunks if c.get("clip_candidate"))
        print(f"  Processed {i+1}/{len(chunks)} ({candidates} candidates)")

# Summary
candidates = [c for c in processed_chunks if c.get("clip_candidate")]
high_scores = [c for c in processed_chunks if c.get("emotion_score", 0) >= 8]

print("\n" + "="*60)
print("PROCESSING COMPLETE")
print("="*60)
print(f"Total: {len(chunks)}, Skipped: {skipped_count}")
print(f"Clip candidates (≥7): {len(candidates)}")
print(f"High priority (≥8): {len(high_scores)}")
print("="*60)

## Step 7: Merge Adjacent Clips

In [ ]:
def parse_timestamp(ts):
    if not ts: return 0.0
    parts = str(ts).replace(".", ":").split(":")
    if len(parts) == 2: return float(parts[0]) * 60 + float(parts[1])
    elif len(parts) == 3: return float(parts[0]) * 3600 + float(parts[1]) * 60 + float(parts[2])
    return 0.0

def merge_adjacent_clips(candidates, max_dur=60, min_dur=15):
    if not candidates: return []
    
    sorted_c = sorted(candidates, key=lambda x: parse_timestamp(x.get("timestamp", x.get("timestamp_start", "00:00"))))
    merged = []
    current = None
    
    for c in sorted_c:
        ts = c.get("timestamp_start") or c.get("timestamp")
        te = c.get("timestamp_end") or (float(ts) + 8 if ts else "00:00")
        start = parse_timestamp(ts)
        end = parse_timestamp(te) if isinstance(te, str) else start + 8
        
        if current is None:
            current = {"video_id": c.get("video_id", "unknown"), "timestamp_start": ts, "timestamp_end": te, "start_seconds": start, "end_seconds": end, "topics": [c.get("topic")], "emotion_scores": [c.get("emotion_score", 0)], "chunks": [c]}
        else:
            if start - current["end_seconds"] <= 10:
                current["end_seconds"] = end
                current["topics"].append(c.get("topic"))
                current["emotion_scores"].append(c.get("emotion_score", 0))
                current["chunks"].append(c)
            else:
                dur = current["end_seconds"] - current["start_seconds"]
                if min_dur <= dur <= max_dur:
                    topic_counts = {t: current["topics"].count(t) for t in set(current["topics"])}
                    merged.append({"video_id": current["video_id"], "timestamp_start": current["timestamp_start"], "timestamp_end": current["timestamp_end"], "start_seconds": current["start_seconds"], "end_seconds": current["end_seconds"], "duration": dur, "topic": max(topic_counts, key=topic_counts.get), "all_topics": list(set(current["topics"])), "emotion_score": max(current["emotion_scores"]), "chunk_count": len(current["chunks"])})
                current = {"video_id": c.get("video_id", "unknown"), "timestamp_start": ts, "timestamp_end": te, "start_seconds": start, "end_seconds": end, "topics": [c.get("topic")], "emotion_scores": [c.get("emotion_score", 0)], "chunks": [c]}
    
    if current:
        dur = current["end_seconds"] - current["start_seconds"]
        if min_dur <= dur <= max_dur:
            topic_counts = {t: current["topics"].count(t) for t in set(current["topics"])}
            merged.append({"video_id": current["video_id"], "timestamp_start": current["timestamp_start"], "timestamp_end": current["timestamp_end"], "start_seconds": current["start_seconds"], "end_seconds": current["end_seconds"], "duration": dur, "topic": max(topic_counts, key=topic_counts.get), "all_topics": list(set(current["topics"])), "emotion_score": max(current["emotion_scores"]), "chunk_count": len(current["chunks"])})
    
    return merged


# Merge clips
if processed_chunks:
    candidates = [c for c in processed_chunks if c.get("clip_candidate")]
    merged = merge_adjacent_clips(candidates)
    
    print(f"\n✓ Merged {len(candidates)} candidates into {len(merged)} clips")
    print("\nTop clips:")
    for i, clip in enumerate(sorted(merged, key=lambda x: -x['emotion_score'])[:10], 1):
        print(f"  {i:2d}. [{clip['topic']:12}] {clip['timestamp_start']}-{clip['timestamp_end']} | Score: {clip['emotion_score']} | {clip['duration']:.0f}s")

## Step 8: Save Results

In [ ]:
from datetime import datetime

if merged:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Save JSON
    json_file = f"/kaggle/working/clip_candidates_qwen35_4b_{timestamp}.json"
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(merged, f, indent=2, ensure_ascii=False)
    print(f"✓ Saved JSON: {json_file}")
    print(f"  Size: {os.path.getsize(json_file) / 1024:.1f} KB")
    
    # Save CSV
    csv_file = f"/kaggle/working/clip_candidates_qwen35_4b_{timestamp}.csv"
    with open(csv_file, 'w', encoding='utf-8') as f:
        f.write("video_id,topic,start_time,end_time,duration,emotion_score\n")
        for clip in merged:
            f.write(f"\"{clip['video_id']}\",\"{clip['topic']}\",\"{clip['timestamp_start']}\",\"{clip['timestamp_end']}\",{clip['duration']:.0f},{clip['emotion_score']}\n")
    print(f"✓ Saved CSV: {csv_file}")
    
    # Show download links
    print("\n" + "="*60)
    print("DOWNLOAD FILES")
    print("="*60)
    print(f"JSON: {json_file}")
    print(f"CSV: {csv_file}")
    print("\nClick on the file icons (left panel) to download!")
    
    # Show ffmpeg commands
    print("\n" + "="*60)
    print("FFMPEG COMMANDS (Run locally)")
    print("="*60)
    for i, clip in enumerate(merged[:15], 1):
        topic = clip['topic'].lower().replace(' ', '_')
        print(f"# Clip {i:2d}: {clip['topic']} (Score: {clip['emotion_score']})")
        print(f"ffmpeg -ss {clip['timestamp_start']} -to {clip['timestamp_end']} -i video.mp4 clip_{i:02d}_{topic}.mp4\n")